### Extracting Clinical Records Text into into delta table

In [0]:
import importlib
import sys
sys.path.append(
    '/Workspace/ATCI_CLINICAL_TRANSCRIPT_AI_HACKATHON/Utils'
)
import clinical_extractor
importlib.reload(clinical_extractor)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime, timezone

# Created Strcture of table
schema = StructType([
    StructField("patient_name", StringType()),
    StructField("mrn", StringType()),
    StructField("dob", StringType()),
    StructField("visit_date", StringType()),
    StructField("provider", StringType()),
    StructField("reason", StringType()),
    StructField("subjective", StringType()),
    StructField("objective", StringType()),
    StructField("bp_systolic", IntegerType()),
    StructField("bp_diastolic", IntegerType()),
    StructField("heart_rate", IntegerType()),
    StructField("assessment_plan", ArrayType(StringType()))
])

extract_udf = udf(clinical_extractor.extract_clinical_fields, schema)


In [0]:

# parsing data records
df = spark.table("atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_extracted_bronze")

df_parsed = df.withColumn(
    "clinical_parsed_data",
    extract_udf("page_text")
)

df_parsed = df_parsed.withColumn(
    "update_datetime",
    lit(datetime.now(timezone.utc).isoformat(timespec="seconds"))
)

df_parsed = df_parsed.withColumn(
    "updated_by",
    lit(spark.sql("SELECT current_user()").collect()[0][0])
)


In [0]:
display(df_parsed)

In [0]:
display(df_parsed.select(
    "file_name",
    "page_number",
    "clinical_parsed_data.patient_name",
    "clinical_parsed_data.mrn",
    "clinical_parsed_data.dob",
    "clinical_parsed_data.visit_date",
    "clinical_parsed_data.provider",
    "clinical_parsed_data.reason",
    "clinical_parsed_data.subjective",
    "clinical_parsed_data.objective",
    "clinical_parsed_data.bp_systolic",
    "clinical_parsed_data.bp_diastolic",
    "clinical_parsed_data.heart_rate",
    "clinical_parsed_data.assessment_plan"
))

In [0]:
# Flattening records
df_final = df_parsed.select(
    "file_name",
    "page_number",
    "clinical_parsed_data.patient_name",
    "clinical_parsed_data.mrn",
    "clinical_parsed_data.dob",
    "clinical_parsed_data.visit_date",
    "clinical_parsed_data.provider",
    "clinical_parsed_data.reason",
    "clinical_parsed_data.subjective",
    "clinical_parsed_data.objective",
    "clinical_parsed_data.bp_systolic",
    "clinical_parsed_data.bp_diastolic",
    "clinical_parsed_data.heart_rate",
    "clinical_parsed_data.assessment_plan",
    "update_datetime",
    "updated_by"
)

# Writing data to silver table
df_final.write.mode("overwrite").saveAsTable("atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_parsed_silver")


In [0]:
silver_tb = spark.table("atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_parsed_silver")

In [0]:
silver_tb.summary().display()

In [0]:
from pyspark.sql import functions as F
display(silver_tb.filter(F.length(F.col("objective")) > 18))

#### **Extracting Data from Parsed data in Bronze Layer is Successfully completed**